# Module P06 — Cas limites : collision, hors service, zones X, météo

**Objectif :** traiter les situations exceptionnelles qui rendent le jeu robuste.

> Compatible Google Colab — aucun fichier externe requis.

## Section 1 — Rappels synthétiques

### Ce qu'on a vu dans P05 (prérequis)
- `executer_mouvement()` gère le chemin nominal : déplacement, batterie, prise, livraison
- Toujours mettre à jour **dictionnaire drone** ET **grille** ensemble

### Cas limites à couvrir dans P06
| Cas | Déclencheur | Effet |
|-----|------------|-------|
| Zone X | `(col, lig) in zones_x` | batterie -2 supplémentaire |
| Batterie épuisée | `batterie <= 0` | `hors_service = True` |
| Collision tempête | même case après déplacement | `bloque = 2` |
| Phase météo | fin de tour | tempêtes bougent (50%) |
| Propagation zones X | tous les N tours | nouvelles cases X |
| Fin de partie | conditions | `partie_finie = True` |

### La règle du `set` pour les zones X
```python
# set → vérification O(1) (instantané quelle que soit la taille)
(col, lig) in etat['zones_x']   # True ou False

# list → vérification O(n) (ralentit sur grande grille)
(col, lig) in etat['zones_x']   # pareil en syntaxe, mais plus lent
```

In [ ]:
import random
random.seed(42)  # reproductibilité des tests
GRILLE_TAILLE = 12
BATTERIE_MAX = 20

def creer_etat_test(batterie=15, survivant_embarque=False, bloque=0,
                    hors_service=False, zones_x=None):
    """Crée un état minimal configurable pour les tests P06."""    grille = [['.' for _ in range(12)] for _ in range(12)]
    etat = {
        'tour': 1, 'score': 0, 'partie_finie': False, 'victoire': False,
        'grille': grille, 'hopital': (0, 0), 'batiments': [],
        'drones': {
            'D1': {'id': 'D1', 'col': 3, 'lig': 3,
                   'batterie': batterie, 'batterie_max': BATTERIE_MAX,
                   'survivant': 'S1' if survivant_embarque else None,
                   'bloque': bloque, 'hors_service': hors_service}
        },
        'tempetes': {
            'T1': {'id': 'T1', 'col': 5, 'lig': 5, 'dx': 1, 'dy': 0}
        },
        'survivants': {
            'S1': {'id': 'S1', 'col': 4, 'lig': 3, 'etat': 'en_attente'}
        },
        'zones_x': zones_x if zones_x else set(),
        'historique': []
    }
    etat['grille'][0][0] = 'H'
    etat['grille'][3][3] = 'D1'
    etat['grille'][3][4] = 'S1'
    etat['grille'][5][5] = 'T1'
    return etat

print('Fonctions utilitaires chargées ✓')

## Section 2 — Exercices guidés

### Étape 1 — Zone X : surcoût batterie
Si la case cible est dans `etat['zones_x']`, la batterie perd 2 points supplémentaires.
Après déduction, si `drone['batterie'] <= 0`, le drone est `hors_service`.

In [ ]:
def appliquer_zone_x(etat, id_drone, col_cible, lig_cible):
    """Applique le surcoût zone X et gère hors_service si batterie <= 0."""    drone = etat['drones'][id_drone]

    # TODO 1 : vérifier si (col_cible, lig_cible) est dans etat['zones_x']
    # Si oui : drone['batterie'] -= 2
    # if ...:
    #     ...

    # TODO 2 : si batterie <= 0 après déduction, mettre hors_service
    # if drone['batterie'] <= 0:
    #     drone['batterie'] = 0
    #     drone['hors_service'] = True
    #     return f"{id_drone} HORS SERVICE"

    return ''

In [ ]:
# Test zone X — batterie 3, zone X sur case cible
etat = creer_etat_test(batterie=3, zones_x={(4, 3)})
etat['grille'][3][4] = 'X'

# Simuler qu'un déplacement nominal a déjà coûté 1
etat['drones']['D1']['batterie'] -= 1  # batterie = 2 après déplacement

log = appliquer_zone_x(etat, 'D1', 4, 3)
print('Log :', log)

assert etat['drones']['D1']['batterie'] == 0, f"batterie attendue 0, obtenu {etat['drones']['D1']['batterie']}"
assert etat['drones']['D1']['hors_service'] == True, 'D1 doit être hors service'
print('✓ Étape 1 validée')

### Étape 2 — Collision drone-tempête (blocage 2 tours)

In [ ]:
def verifier_collision(etat, id_drone):
    """Vérifie si le drone est sur une tempête après déplacement."""    drone = etat['drones'][id_drone]

    # TODO : parcourir etat['tempetes']
    # Si une tempête a les mêmes col et lig que le drone :
    #   drone['bloque'] = 2
    #   return f"{id_drone} BLOQUÉ par {tid}"
    # for tid, tempete in etat['tempetes'].items():
    #     if ...:
    #         ...

    return ''

In [ ]:
# Test collision : placer D1 sur la même case que T1
etat = creer_etat_test()
etat['drones']['D1']['col'] = 5
etat['drones']['D1']['lig'] = 5

log = verifier_collision(etat, 'D1')
print('Log :', log)

assert etat['drones']['D1']['bloque'] == 2, 'D1 doit être bloqué 2 tours'
print('✓ Étape 2 validée')

### Étape 3 — Phase météo : déplacer les tempêtes (50% de chance)

In [ ]:
def deplacer_tempetes(etat):
    """Phase météo : chaque tempête a 50% de chance de se déplacer."""    TAILLE = 12
    for tid, t in etat['tempetes'].items():
        # TODO : si random.random() < 0.5 :
        #   - effacer l'ancienne position sur la grille
        #   - calculer nouvelle col : max(0, min(TAILLE-1, t['col'] + t['dx']))
        #   - calculer nouvelle lig : max(0, min(TAILLE-1, t['lig'] + t['dy']))
        #   - mettre à jour t['col'] et t['lig']
        #   - marquer la nouvelle position sur la grille
        pass

In [ ]:
# Test phase météo — forcer le déplacement avec random fixé
random.seed(0)  # avec seed 0, random.random() < 0.5 au 1er appel
etat = creer_etat_test()
col_avant = etat['tempetes']['T1']['col']  # 5
lig_avant = etat['tempetes']['T1']['lig']  # 5

deplacer_tempetes(etat)
col_apres = etat['tempetes']['T1']['col']
lig_apres = etat['tempetes']['T1']['lig']
print(f'T1 : ({col_avant},{lig_avant}) → ({col_apres},{lig_apres})')
# Avec seed 0, T1 devrait avoir bougé de dx=1 vers (6, 5)
# (résultat dépend du seed — vérifier manuellement)
print('Phase météo exécutée ✓')

### Étape 4 — Vérification de fin de partie

In [ ]:
def verifier_fin_partie(etat, TOURS_MAX=50):
    """Vérifie les 3 conditions de fin de partie."""    survivants = etat['survivants']
    drones = etat['drones']

    # TODO 1 : victoire si tous les survivants ont etat == 'sauve'
    # if all(s['etat'] == 'sauve' for s in survivants.values()):
    #     etat['partie_finie'] = True
    #     etat['victoire'] = True
    #     return 'Victoire !'

    # TODO 2 : défaite si tous les drones sont hors_service
    # if all(d['hors_service'] for d in drones.values()):
    #     ...

    # TODO 3 : défaite si tours >= TOURS_MAX
    # if etat['tour'] >= TOURS_MAX:
    #     ...

    return ''

In [ ]:
# Test 1 : victoire
etat = creer_etat_test()
etat['survivants']['S1']['etat'] = 'sauve'
msg = verifier_fin_partie(etat)
assert etat['partie_finie'] == True, 'partie doit être finie'
assert etat['victoire'] == True, 'victoire attendue'
print('Test victoire OK :', msg)

# Test 2 : défaite — tous drones hors service
etat2 = creer_etat_test()
etat2['drones']['D1']['hors_service'] = True
msg2 = verifier_fin_partie(etat2)
assert etat2['partie_finie'] == True
assert etat2['victoire'] == False
print('Test défaite drones OK :', msg2)

# Test 3 : défaite — tours max
etat3 = creer_etat_test()
etat3['tour'] = 50
msg3 = verifier_fin_partie(etat3)
assert etat3['partie_finie'] == True
assert etat3['victoire'] == False
print('Test défaite tours OK :', msg3)
print('\n✓ Module P06 complet !')